# 01 - Persiapan Data untuk Fine-tuning IndoBERTweet

Notebook ini **belum melatih model**. Tugasnya menyiapkan dataset final, untuk **dua skema label sekaligus**:

- **binary**: `negatif` vs `non-negatif`
- **3class**: `negatif` / `netral` / `positif`

Split (train/val/test) dan fold (5-fold CV) sama persis di kedua skema — perbedaannya cuma di `label`/`label_name`. Dengan begini S0/S1/S3/S5 untuk binary dan 3class diuji di baris komentar yang sama, hasilnya bisa dibandingkan adil.

Langkah:
1. Gabungkan komentar yang **agree** (Shafwa = Khalila) dan yang **disagree** (sudah diputuskan di `disagreements_label_final.csv`)
2. Normalisasi teks sesuai konvensi IndoBERTweet (`@USER`, `HTTPURL`, emoji jadi teks)
3. Split sekali (stratified by label 3-kelas, supaya proporsi negatif/netral/positif terjaga di kedua skema)
4. Simpan dua folder: `data/siap_latih/binary/` dan `data/siap_latih/3class/`

Install dulu kalau belum ada:
```
pip install pandas scikit-learn emoji transformers torch
```

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import emoji
import re
from sklearn.model_selection import StratifiedKFold, StratifiedGroupKFold
from sklearn.utils.class_weight import compute_class_weight

# ============================ KONFIGURASI ============================
SPLIT_BY          = 'comment'  # 'comment' | 'influencer' (satu influencer tidak terpecah antar split)
SEED              = 42
N_FOLDS           = 5          # untuk cross-validation
N_SPLIT_PARTS     = 7          # 7 bagian -> test ~14%, val ~14%, train ~72%
USE_S1_PREPROCESS = True       # True: @USER / HTTPURL / emoji->teks ; False: pakai comment_text_clean apa adanya
MODEL_NAME        = 'indolem/indobertweet-base-uncased'
# =====================================================================

# Path relatif dari lokasi notebook (02_IndoBert/03_finetunning_model)
ROOT_DIR     = Path.cwd().parent
LABELING_DIR = ROOT_DIR / 'labeling'
DATA_DIR     = ROOT_DIR / 'data'

PATH_SHAFWA  = LABELING_DIR / 'labels_shafwa.csv'
PATH_KHALILA = LABELING_DIR / 'labels_khalila.csv'
PATH_DISAGREE = DATA_DIR / 'disagreements_label_final.csv'

OUT_DIR = DATA_DIR / 'siap_latih'
OUT_DIR.mkdir(parents=True, exist_ok=True)

for p in [PATH_SHAFWA, PATH_KHALILA, PATH_DISAGREE]:
    print(('OK      ' if p.exists() else 'HILANG  ') + p.name)

OK      labels_shafwa.csv
OK      labels_khalila.csv
OK      disagreements_label_final.csv


## 1. Load dua labeler dan sejajarkan

Dua file labeler diurutkan sama, jadi disejajarkan per baris (bukan merge by teks).
Merge by teks sebelumnya menghasilkan 2004 baris karena ada teks duplikat, dan tidak membawa `username` / `video_id`.
Kalau urutan ternyata beda, cell ini akan berhenti dengan error.

In [2]:
s = pd.read_csv(PATH_SHAFWA)
k = pd.read_csv(PATH_KHALILA)
assert len(s) == len(k), f'Jumlah baris beda: {len(s)} vs {len(k)}'

# Pastikan baris ke-i di kedua file memang komentar yang sama
same_row = (
    (s['video_id'].astype(str).values == k['video_id'].astype(str).values)
    & (s['comment_text_raw'].fillna('').values == k['comment_text_raw'].fillna('').values)
)
assert same_row.all(), f'{(~same_row).sum()} baris tidak sejajar antara dua file'

df = s[['username', 'video_id', 'comment_text_raw', 'comment_text_clean']].copy()
df['label_shafwa']  = s['label_manual'].str.lower().str.strip()
df['label_khalila'] = k['label_manual'].str.lower().str.strip()

print('Baris:', len(df))
print('Label kosong:', df[['label_shafwa', 'label_khalila']].isna().sum().to_dict())
print('Teks kosong :', df['comment_text_clean'].isna().sum())

Baris: 2000
Label kosong: {'label_shafwa': 0, 'label_khalila': 0}
Teks kosong : 0


## 2. Gabungkan agree + disagree

- Agree: `label_final` = label yang disepakati
- Disagree: `label_final` diambil dari `disagreements_label_final.csv` (kunci: teks + label Shafwa + label Khalila, supaya teks duplikat dengan label berbeda tidak tertukar)

`label_final` ini yang jadi kelas 3-kelas (`negatif`/`netral`/`positif`); skema binary diturunkan darinya di langkah 4.

In [3]:
dis = pd.read_csv(PATH_DISAGREE)
dis['label_shafwa']  = dis['label_shafwa'].str.lower().str.strip()
dis['label_khalila'] = dis['label_khalila'].str.lower().str.strip()
dis['label_final']   = dis['label_final'].str.lower().str.strip()

key = ['comment_text_clean', 'label_shafwa', 'label_khalila']

# file disagreement bisa memuat baris dobel (efek merge lama) -> pastikan keputusannya konsisten
conflict = dis.groupby(key)['label_final'].nunique()
assert (conflict == 1).all(), 'Ada komentar yang sama tapi label_final berbeda'
dis_map = dis.drop_duplicates(key)[key + ['label_final']]

df = df.merge(dis_map, on=key, how='left', validate='many_to_one')

is_agree = df['label_shafwa'] == df['label_khalila']
df.loc[is_agree, 'label_final'] = df.loc[is_agree, 'label_shafwa']
df['label_source'] = np.where(is_agree, 'agree', 'resolved')

missing = df['label_final'].isna().sum()
assert missing == 0, f'{missing} komentar disagreement belum punya label_final'

# Aturan: hasil resolusi harus salah satu label labeler, atau netral
ok = (df['label_final'].eq(df['label_shafwa']) | df['label_final'].eq(df['label_khalila'])
      | df['label_final'].eq('netral'))
assert ok.all(), 'Ada label_final yang melanggar aturan'

print(df['label_source'].value_counts().to_string())
print()
print(pd.crosstab(df['label_final'], df['label_source'], margins=True))

label_source
agree       1670
resolved     330

label_source  agree  resolved   All
label_final                        
negatif          29        20    49
netral         1497       287  1784
positif         144        23   167
All            1670       330  2000


## 3. Normalisasi teks (konvensi IndoBERTweet)

IndoBERTweet dilatih dengan `@USER`, `HTTPURL`, dan emoji yang diubah jadi teks. Kolom `text_model` dipakai sebagai input model.
Set `USE_S1_PREPROCESS = False` untuk memakai `comment_text_clean` apa adanya.

In [4]:
URL_RE  = re.compile(r'(https?://\S+|www\.\S+)')
USER_RE = re.compile(r'@\w+')

def normalize_tweet(text: str) -> str:
    text = '' if pd.isna(text) else str(text)
    text = URL_RE.sub('HTTPURL', text)
    text = USER_RE.sub('@USER', text)
    text = emoji.demojize(text, language='id')       # 😭 -> :wajah_menangis_keras:
    text = re.sub(r':([^:\s]+):', r' \1 ', text)      # buang titik dua pembungkus
    text = text.replace('_', ' ')
    text = re.sub(r'\s+', ' ', text).strip()
    return text.lower()

SOURCE_TEXT = 'comment_text_raw'   # raw masih punya @mention / URL; clean dipakai kalau raw kosong
base = df[SOURCE_TEXT].where(df[SOURCE_TEXT].notna() & df[SOURCE_TEXT].astype(str).str.strip().ne(''),
                             df['comment_text_clean'])

if USE_S1_PREPROCESS:
    df['text_model'] = base.map(normalize_tweet)
else:
    df['text_model'] = df['comment_text_clean'].fillna('').astype(str).str.strip().str.lower()

empty = df['text_model'].str.len() == 0
print('Teks kosong setelah normalisasi:', empty.sum())
df = df[~empty].reset_index(drop=True)

for i in df.sample(5, random_state=SEED).index:
    print('RAW  :', df.loc[i, 'comment_text_raw'])
    print('MODEL:', df.loc[i, 'text_model'])
    print()

Teks kosong setelah normalisasi: 0
RAW  : ka dipunggung sama dada aku ada jerawat pakai bedak dingin yg ada kandungan sulfur sama pakai sabun acneway yg ada kandungan sulfur jadi kering banget, ngelupas dan agak merah tolong ka gmna ? biar lembab, sama ilang bekas jerawatnya gatal juga kadang²
MODEL: ka dipunggung sama dada aku ada jerawat pakai bedak dingin yg ada kandungan sulfur sama pakai sabun acneway yg ada kandungan sulfur jadi kering banget, ngelupas dan agak merah tolong ka gmna ? biar lembab, sama ilang bekas jerawatnya gatal juga kadang²

RAW  : yang panas kak, biar semakin menyala
MODEL: yang panas kak, biar semakin menyala

RAW  : buah tetap dimakan kak, jgn d skip kec gula buatan
MODEL: buah tetap dimakan kak, jgn d skip kec gula buatan

RAW  : iyakah ka? aku pas lebaran malan soto sama rendang ga tau ternyata pedes, ke ada rasa perih dari dalem perut aku mikirnya kena luka yang didalem kali ya jadi agak takut kenapa² gituloh ka hehe
MODEL: iyakah ka? aku pas lebaran mala

## 4. Split data (dipakai bersama oleh binary & 3class)

Split distratifikasi berdasarkan `label_final` (3 kelas), supaya proporsi negatif/netral/positif terjaga di train/val/test — otomatis juga menjaga proporsi binary.

- `split` (train / val / test): untuk satu kali eksperimen hold-out (S0, S1, S3, S5 dibandingkan di **test yang sama**, untuk **kedua skema label**)
- `fold` (0-4): untuk cross-validation

In [5]:
def assign_parts(data, n_parts, seed, strat_col):
    if SPLIT_BY == 'influencer':
        cv = StratifiedGroupKFold(n_splits=n_parts, shuffle=True, random_state=seed)
        it = cv.split(data, data[strat_col], groups=data['username'])
    else:
        cv = StratifiedKFold(n_splits=n_parts, shuffle=True, random_state=seed)
        it = cv.split(data, data[strat_col])
    part = np.full(len(data), -1)
    for i, (_, te) in enumerate(it):
        part[te] = i
    return part

parts = assign_parts(df, N_SPLIT_PARTS, SEED, 'label_final')
df['split'] = np.select([parts == 0, parts == 1], ['test', 'val'], default='train')
df['fold']  = assign_parts(df, N_FOLDS, SEED, 'label_final')

print('--- split (3 kelas) ---')
print(pd.crosstab(df['split'], df['label_final'], margins=True))
print()
print('--- fold (3 kelas) ---')
print(pd.crosstab(df['fold'], df['label_final'], margins=True))

--- split (3 kelas) ---
label_final  negatif  netral  positif   All
split                                      
test               7     255       24   286
train             35    1274      119  1428
val                7     255       24   286
All               49    1784      167  2000

--- fold (3 kelas) ---
label_final  negatif  netral  positif   All
fold                                       
0                 10     357       33   400
1                 10     357       33   400
2                 10     357       33   400
3                  9     357       34   400
4                 10     356       34   400
All               49    1784      167  2000


In [6]:
# Peringatan kalau kelas minoritas terlalu sedikit untuk evaluasi yang stabil
n_neg_test = ((df['split'] == 'test') & (df['label_final'] == 'negatif')).sum()
n_neg_all  = (df['label_final'] == 'negatif').sum()
print(f'Negatif total: {n_neg_all} | di test: {n_neg_test}')
if n_neg_test < 30:
    print('PERHATIAN: komentar negatif di test < 30, F1-negatif sangat tidak stabil.')
    print('Lebih aman melaporkan hasil cross-validation (kolom fold) dan/atau menambah data negatif.')

Negatif total: 49 | di test: 7
PERHATIAN: komentar negatif di test < 30, F1-negatif sangat tidak stabil.
Lebih aman melaporkan hasil cross-validation (kolom fold) dan/atau menambah data negatif.


## 5. Siapkan dua skema label dan simpan masing-masing

Dibuat dua salinan dataset dengan `split`/`fold` yang identik, hanya `label`/`label_name` yang beda:

- `data/siap_latih/binary/`
- `data/siap_latih/3class/`

In [7]:
SCHEMES = {
    'binary': {
        'label_names': ['non-negatif', 'negatif'],
        'label_name_col': np.where(df['label_final'] == 'negatif', 'negatif', 'non-negatif'),
    },
    '3class': {
        'label_names': ['negatif', 'netral', 'positif'],
        'label_name_col': df['label_final'],
    },
}

cols_full = ['username', 'video_id', 'comment_text_raw', 'comment_text_clean', 'text_model',
             'label_shafwa', 'label_khalila', 'label_final', 'label_source',
             'label_name', 'label', 'split', 'fold']

summary = {}
for scheme_name, cfg in SCHEMES.items():
    d = df.copy()
    label_names = cfg['label_names']
    label2id = {n: i for i, n in enumerate(label_names)}
    id2label = {i: n for n, i in label2id.items()}
    d['label_name'] = cfg['label_name_col']
    d['label'] = d['label_name'].map(label2id)

    scheme_dir = OUT_DIR / scheme_name
    scheme_dir.mkdir(parents=True, exist_ok=True)

    d[cols_full].to_csv(scheme_dir / 'dataset_final.csv', index=False)
    for part in ['train', 'val', 'test']:
        d.loc[d['split'] == part, ['text_model', 'label', 'label_name', 'username', 'video_id']] \
          .to_csv(scheme_dir / f'{part}.csv', index=False)

    train = d[d['split'] == 'train']
    classes = np.array(sorted(d['label'].unique()))
    w = compute_class_weight('balanced', classes=classes, y=train['label'])
    class_weights = {int(c): float(x) for c, x in zip(classes, w)}

    dist = d['label_name'].value_counts().reindex(label_names)
    summary[scheme_name] = {'label2id': label2id, 'class_weights': class_weights}

    print(f'=== {scheme_name} ===')
    print(pd.DataFrame({'jumlah': dist, 'persen': (dist / len(d) * 100).round(1)}))
    print('class_weight:', {id2label[c]: round(x, 3) for c, x in class_weights.items()})
    print()

=== binary ===
             jumlah  persen
label_name                 
non-negatif    1951    97.6
negatif          49     2.4
class_weight: {'non-negatif': 0.513, 'negatif': 20.4}

=== 3class ===
            jumlah  persen
label_name                
negatif         49     2.4
netral        1784    89.2
positif        167     8.4
class_weight: {'negatif': 13.6, 'netral': 0.374, 'positif': 4.0}



## 6. Panjang token (`MAX_LEN`)

Dihitung sekali dari `text_model` (sama untuk kedua skema). Butuh download tokenizer dari HuggingFace, jadi perlu internet pada run pertama.

In [8]:
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained(MODEL_NAME)
lens = df['text_model'].map(lambda t: len(tok.encode(t, add_special_tokens=True)))
print(lens.describe().round(1).to_string())

p95 = int(np.ceil(np.percentile(lens, 95)))
MAX_LEN = int(min(128, int(np.ceil(p95 / 8) * 8)))
print(f'\nP95 = {p95} token -> MAX_LEN = {MAX_LEN}')
print(f'Komentar terpotong pada MAX_LEN: {(lens > MAX_LEN).mean() * 100:.1f}%')

c:\Users\nesha\Tugas_akhir\seleksi-influencer\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


count    2000.0
mean       20.6
std        18.4
min         4.0
25%        10.0
50%        16.0
75%        25.0
max       215.0

P95 = 51 token -> MAX_LEN = 56
Komentar terpotong pada MAX_LEN: 3.6%


## 7. Simpan config per skema

In [9]:
for scheme_name, cfg in SCHEMES.items():
    config = {
        'model_name': MODEL_NAME,
        'label_scheme': scheme_name,
        'split_by': SPLIT_BY,
        'seed': SEED,
        'use_s1_preprocess': USE_S1_PREPROCESS,
        'label2id': summary[scheme_name]['label2id'],
        'id2label': {str(v): k_ for k_, v in summary[scheme_name]['label2id'].items()},
        'class_weights': summary[scheme_name]['class_weights'],
        'max_len': MAX_LEN,
        'n_total': int(len(df)),
    }
    out = OUT_DIR / scheme_name / 'config.json'
    out.write_text(json.dumps(config, indent=2, ensure_ascii=False), encoding='utf-8')
    print('Tersimpan:', out)

print()
print('Isi data/siap_latih:')
for p in sorted(OUT_DIR.rglob('*')):
    if p.is_file():
        print(' -', p.relative_to(OUT_DIR))

Tersimpan: c:\Users\nesha\Tugas_akhir\seleksi-influencer\02_IndoBert\data\siap_latih\binary\config.json
Tersimpan: c:\Users\nesha\Tugas_akhir\seleksi-influencer\02_IndoBert\data\siap_latih\3class\config.json

Isi data/siap_latih:
 - 3class\config.json
 - 3class\dataset_final.csv
 - 3class\test.csv
 - 3class\train.csv
 - 3class\val.csv
 - binary\config.json
 - binary\dataset_final.csv
 - binary\test.csv
 - binary\train.csv
 - binary\val.csv
 - config.json
 - dataset_final.csv
 - test.csv
 - train.csv
 - val.csv
